# R6 — MCP

The client's other teams use MCP clients, so the same four R1 tools are served through the Model Context Protocol by the managed Unity Catalog functions server for claims.genai. This notebook lists the tools the server exposes, makes a lookup_claim call for one of the Paper 4 probe claims and a compute_payable call for the same claim whose result equals the verified answer, and records an agent run working with tools it took from that server. Evidence: evidence/r6_mcp.json and agent/traces/r6_mcp_agent.json (same keys as the R2 traces).

In [0]:
# The MCP Python SDK is not in the serverless base image - install it notebook-scoped.
# Pin 1.x: its streamablehttp_client(url, headers=...) is the documented managed-MCP API
# (2.x renames it and drops the headers parameter). Restart so the pinned version is used.
%pip install "mcp>=1.9,<2"
dbutils.library.restartPython()

In [0]:
# Config - catalog, models, paths, the managed MCP server URL, and the probe claim
import os

CATALOG = "claims"
SCHEMA = "genai"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"
REPO_DIR = "/Workspace/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1/Gen AI - Paper 5 - Scenario Based Assignment"
R1_TOOLS = ["lookup_claim", "plan_at", "search_policy", "compute_payable"]

spark.sql(f"USE CATALOG {CATALOG}")

import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
AUTH = w.config.authenticate()["Authorization"]          # Bearer token, never printed or saved
HOST = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
MCP_SERVER = f"{HOST}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}"

PROBE_CLAIM = spark.sql(
    "SELECT claim_id FROM claims.genai.adjudication_probes ORDER BY claim_id LIMIT 1"
).collect()[0].claim_id

print(f"catalog={CATALOG}.{SCHEMA}  chat_model={CHAT_MODEL}")
print(f"managed MCP server: {MCP_SERVER}")
print(f"Paper 4 probe claim: {PROBE_CLAIM}")

In [0]:
# MCP client over Streamable HTTP. Every call authenticates with the same Databricks bearer
# token (never printed); sessions are opened per call. list_tools shows the server's four
# tools with their parameter names; call_tool returns the text the tool produced.
import asyncio
import json
import threading

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


def _run_async(coro):
    """Run a coroutine on a fresh loop (notebooks may already own one)."""
    out = {}

    def runner():
        try:
            out["result"] = asyncio.run(coro)
        except BaseException as e:  # noqa: BLE001 - re-raised below
            out["error"] = e

    t = threading.Thread(target=runner)
    t.start()
    t.join()
    if "error" in out:
        raise out["error"]
    return out["result"]


def mcp_list_tools():
    async def go():
        async with streamablehttp_client(MCP_SERVER, headers={"Authorization": AUTH}) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                resp = await session.list_tools()
                return [{"name": t.name,
                         "description": t.description or "",
                         "parameters": sorted(t.inputSchema.get("properties", {}).keys()),
                         "inputSchema": t.inputSchema}
                        for t in resp.tools]
    return _run_async(go())


def mcp_call_tool(name: str, arguments: dict) -> str:
    async def go():
        async with streamablehttp_client(MCP_SERVER, headers={"Authorization": AUTH}) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                resp = await session.call_tool(name, arguments)
                return "\n".join(c.text for c in resp.content if hasattr(c, "text"))
    return _run_async(go())


def _short(name: str) -> str:
    """Function name after any catalog/schema prefix the server adds."""
    return name.replace(".", "__").split("__")[-1]

specs = mcp_list_tools()
names = [s["name"] for s in specs]
print("server lists", len(names), "tools:", names)
assert sorted(_short(n) for n in names) == sorted(R1_TOOLS), \
    f"expected the four R1 tools, got {names}"
for s in specs:
    print(f"  {s['name']}(" + ", ".join(s["parameters"]) + ")")

In [0]:
# MCP calls: a lookup_claim call for one of the Paper 4 probe claims, and a compute_payable
# call for the same claim whose result must equal the verified answer.
lookup_name = next(n for n in names if _short(n) == "lookup_claim")
payout_name = next(n for n in names if _short(n) == "compute_payable")

lookup_raw = mcp_call_tool(lookup_name, {"claim_id": PROBE_CLAIM})

def parse_tool_result(raw: str) -> list:
    """The managed MCP server returns table results as {columns, rows}; scalar UC functions
    (e.g. compute_payable) return a JSON string - normalise both to a list of objects."""
    val = json.loads(raw)
    if isinstance(val, str):  # double-encoded JSON string from a scalar function
        try:
            val = json.loads(val)
        except json.JSONDecodeError:
            return [val]
    if isinstance(val, dict) and "columns" in val and "rows" in val:
        rows = [dict(zip(val["columns"], r)) for r in val["rows"]]
        if set(val["columns"]) == {"output"} and rows:  # scalar UC function wrapped as {output: "<json>"}
            try:
                inner = json.loads(rows[0]["output"])
                return inner if isinstance(inner, list) else [inner]
            except json.JSONDecodeError:
                return rows
        return rows
    if isinstance(val, list):
        return val
    return [val]

claim = parse_tool_result(lookup_raw)[0]
print(f"lookup_claim through MCP for {PROBE_CLAIM}: plan={claim['plan_code_at_admission']} "
      f"status={claim['status']} billed={claim['billed_amount']}")

facts = {
    "plan_code": claim["plan_code_at_admission"],
    "is_pre_existing": claim["is_pre_existing"],
    "policy_tenure_months": claim["policy_tenure_months"],
    "days_since_policy_start": claim["days_since_policy_start"],
    "is_network_at_admission": claim["is_network_at_admission"],
    "room_rent_per_day": claim["room_rent_per_day"],
    "billed_amount": claim["billed_amount"],
}
payout_raw = mcp_call_tool(payout_name, facts)
payout = parse_tool_result(payout_raw)[0]
print("compute_payable through MCP:", payout_raw)

key = spark.sql(
    f"SELECT * FROM claims.genai.adjudication_verifier WHERE claim_id = '{PROBE_CLAIM}'"
).collect()[0].asDict()
assert payout["payable"] == key["payable"], f"payable differs from the verified answer"
assert payout["waiting_period_ok"] == key["waiting_period_ok"]
assert payout["network_at_admission"] == key["network_at_admission"]
assert abs(payout["room_rent_deduction_pct"] - key["room_rent_deduction_pct"]) < 0.01
assert payout["copay_pct"] == key["copay_pct"]
assert payout["reason_code"] == key["reason_code"], payout["reason_code"]
print(f"PASS: the payout through MCP equals the verified answer for {PROBE_CLAIM}")

In [0]:
# The agent, working with tools it took from the MCP server: the R2 loop, but run_tool is
# replaced by session.call_tool against the managed server. Same evidence shape as R2.
import mlflow

with open(os.path.join(REPO_DIR, "agent", "prompts", "system_v1.md")) as f:
    SYSTEM_PROMPT = f.read()

TOOL_DEFS = [
    {"type": "function", "function": {"name": s["name"], "description": s["description"],
                                      "parameters": s["inputSchema"]}}
    for s in specs
]

def chat(messages, tools):
    body = {"messages": messages, "tools": tools, "temperature": 0, "max_tokens": 1024}
    r = requests.post(
        f"{HOST}/serving-endpoints/{CHAT_MODEL}/invocations",
        headers={"Authorization": AUTH, "Content-Type": "application/json"},
        json=body,
    )
    r.raise_for_status()
    return r.json()

@mlflow.trace(name="saffron_shield_mcp_agent", span_type="AGENT")
def agent_run(question: str, max_steps: int = 4):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    tool_calls_made = []
    for _ in range(max_steps):
        msg = chat(messages, TOOL_DEFS)["choices"][0]["message"]
        tcs = msg.get("tool_calls") or []
        if not tcs:
            mlflow.get_current_active_span().set_attributes({
                "question": question,
                "tool_calls": json.dumps(tool_calls_made),
                "final_answer": msg.get("content") or "",
                "transport": "managed MCP",
            })
            return {"final_answer": msg.get("content") or "",
                    "tool_calls": tool_calls_made,
                    "trace_id": mlflow.get_active_trace_id()}
        messages.append(msg)
        for tc in tcs:
            name = tc["function"]["name"]
            args = json.loads(tc["function"]["arguments"] or "{}")
            result = mcp_call_tool(name, args)          # through the MCP server
            tool_calls_made.append({"name": name, "args": args})
            messages.append({"role": "tool", "tool_call_id": tc["id"], "content": result})
    raise RuntimeError("agent did not settle within max_steps")

QUESTION = f"What's the status of claim {PROBE_CLAIM}?"
mcp_agent_run = agent_run(QUESTION)
print(json.dumps(mcp_agent_run, indent=2, default=str))

_short_names = [_short(tc["name"]) for tc in mcp_agent_run["tool_calls"]]
assert _short_names == ["lookup_claim"], f"expected one lookup_claim through MCP, got {_short_names}"
assert mcp_agent_run["final_answer"].strip()
print(f"agent used the MCP server's tools only: {_short_names}")

In [0]:
# Evidence - evidence/r6_mcp.json and agent/traces/r6_mcp_agent.json, plus grader-style checks:
# managed transport, the server URL, the four tools with their parameter names, the data and
# payout calls whose result equals the verified answer, and a trace that exists in MLflow.
import json
import os

from mlflow.tracking import MlflowClient

r6_evidence = {
    "transport": "managed",
    "server": MCP_SERVER,
    "tools": names,
    "tool_parameters": {s["name"]: s["parameters"] for s in specs},
    "calls": [
        {"tool": lookup_name, "arguments": {"claim_id": PROBE_CLAIM}, "result": lookup_raw},
        {"tool": payout_name, "arguments": facts, "result": payout_raw},
    ],
}

agent_evidence = {
    "trace_id": mcp_agent_run["trace_id"],
    "tool_calls": mcp_agent_run["tool_calls"],
    "final_answer": mcp_agent_run["final_answer"],
    "cited_chunk_ids": [],
}

outputs = [
    (os.path.join(REPO_DIR, "evidence", "r6_mcp.json"), r6_evidence),
    (os.path.join(REPO_DIR, "agent", "traces", "r6_mcp_agent.json"), agent_evidence),
]
for p, rec in outputs:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, "w") as f:
        json.dump(rec, f, indent=2)
    with open(p) as f:
        assert json.load(f) == rec, f"evidence round-trip failed for {p}"
    print(f"wrote {p}")

# grader-style verification
ev = json.load(open(os.path.join(REPO_DIR, "evidence", "r6_mcp.json")))
assert ev["transport"] == "managed"
assert "/api/2.0/mcp/functions/claims/genai" in ev["server"], ev["server"]
assert sorted(_short(n) for n in ev["tools"]) == sorted(R1_TOOLS)
assert "claim_id" in ev["tool_parameters"][lookup_name], "parameter names must be cited"
assert len(ev["calls"]) == 2
assert _short(ev["calls"][0]["tool"]) == "lookup_claim"
assert ev["calls"][0]["arguments"] == {"claim_id": PROBE_CLAIM}
assert parse_tool_result(ev["calls"][0]["result"])[0]["claim_id"] == PROBE_CLAIM
assert _short(ev["calls"][1]["tool"]) == "compute_payable"
_mcp_payout = parse_tool_result(ev["calls"][1]["result"])[0]
assert _mcp_payout["payable"] == key["payable"]
assert _mcp_payout["reason_code"] == key["reason_code"]
assert abs(_mcp_payout["room_rent_deduction_pct"] - key["room_rent_deduction_pct"]) < 0.01

ta = json.load(open(os.path.join(REPO_DIR, "agent", "traces", "r6_mcp_agent.json")))
assert set(ta) == {"trace_id", "tool_calls", "final_answer", "cited_chunk_ids"}
assert [_short(tc["name"]) for tc in ta["tool_calls"]] == ["lookup_claim"]
assert ta["final_answer"].strip()
MlflowClient().get_trace(ta["trace_id"])  # raises if the trace does not exist in MLflow

print(f"\nPASS: r6_mcp.json - managed server at /api/2.0/mcp/functions/claims/genai lists the "
      f"four tools, and the payout through MCP equals the verified answer for {PROBE_CLAIM}; "
      f"agent trace {ta['trace_id']} exists")